In [ ]:
import pandas as pd
import numpy as np

file_clean = "cleaned_equipment_failure_data (1).xlsx"
df_freq = pd.read_excel(file_clean, sheet_name="freq_clean")
df_sev  = pd.read_excel(file_clean, sheet_name="sev_clean")

KEYS = ["policy_id", "equipment_id"]

for df in (df_freq, df_sev):
    if "equipment_age" in df.columns:
        df["equipment_age"] = pd.to_numeric(df["equipment_age"], errors="coerce")
        df.loc[df["equipment_age"] > 10, "equipment_age"] = np.nan


sev_age_lookup = (
    df_sev.dropna(subset=KEYS)
          .groupby(KEYS, as_index=False)["equipment_age"]
          .first()
          .rename(columns={"equipment_age": "equipment_age_from_sev"})
)

freq_age_lookup = (
    df_freq.dropna(subset=KEYS)
           .groupby(KEYS, as_index=False)["equipment_age"]
           .first()
           .rename(columns={"equipment_age": "equipment_age_from_freq"})
)


# freq <- sev
df_freq = df_freq.merge(sev_age_lookup, on=KEYS, how="left")
df_freq["equipment_age"] = df_freq["equipment_age"].fillna(df_freq["equipment_age_from_sev"])
df_freq = df_freq.drop(columns=["equipment_age_from_sev"])


df_sev = df_sev.merge(freq_age_lookup, on=KEYS, how="left")
df_sev["equipment_age"] = df_sev["equipment_age"].fillna(df_sev["equipment_age_from_freq"])
df_sev = df_sev.drop(columns=["equipment_age_from_freq"])

df_freq_aug = df_freq.copy()


output_file = "cleaned_equipment_failure_data_v2.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_freq_aug.to_excel(writer, sheet_name="freq_final", index=False)
    df_sev.to_excel(writer, sheet_name="sev_final", index=False)

print("Selesai. File:", output_file)
print("freq rows:", len(df_freq_aug))
print("sev rows:", len(df_sev))

Selesai. File: cleaned_equipment_failure_data_v2.xlsx
freq rows: 95062
sev rows: 8272


In [ ]:
import pandas as pd
import numpy as np

KEYS = ["policy_id", "equipment_id"]

# =========================
# 1) Hitung claim_count dari sev:
#    jumlah baris per (policy_id, equipment_id)
# =========================
sev_claim_count = (
    df_sev.dropna(subset=KEYS)
          .groupby(KEYS)
          .size()
          .reset_index(name="claim_count_new")
)

# =========================
# 2) Merge ke freq & overwrite claim_count
# =========================
df_freq_aug = df_freq_aug.merge(sev_claim_count, on=KEYS, how="left")

# kalau tidak ada baris di sev untuk key tsb => claim_count = 0
df_freq_aug["claim_count_new"] = df_freq_aug["claim_count_new"].fillna(0).astype(int)

# overwrite claim_count lama (atau buat baru)
df_freq_aug["claim_count"] = df_freq_aug["claim_count_new"]

# drop kolom helper
df_freq_aug = df_freq_aug.drop(columns=["claim_count_new"])

# =========================
# 3) Quick checks
# =========================
print("Total claim_count (freq):", int(df_freq_aug["claim_count"].sum()))
print("Total baris sev yang dipakai:", int(len(df_sev.dropna(subset=KEYS))))

print("\nclaim_count distribution:")
print(df_freq_aug["claim_count"].value_counts().sort_index())

Total claim_count (freq): 8188
Total baris sev yang dipakai: 8235

claim_count distribution:
claim_count
0    88104
1     6580
3        7
4      350
8        2
9       19
Name: count, dtype: int64


In [ ]:
df_freq_aug["claim_count"].unique()

array([0, 1, 4, 3, 9, 8])

In [ ]:
# =========================
# 4) Hapus baris dengan exposure NaN
# =========================
rows_freq_before = len(df_freq_aug)
rows_sev_before = len(df_sev)

df_freq_aug = df_freq_aug.dropna(subset=["exposure"])
df_sev = df_sev.dropna(subset=["exposure"])

print("Baris freq dihapus karena exposure NaN:", rows_freq_before - len(df_freq_aug))
print("Baris sev dihapus karena exposure NaN:", rows_sev_before - len(df_sev))


# =========================
# 5) Hapus baris di sev jika claim_amount NaN
# =========================
rows_sev_before_claim = len(df_sev)

df_sev = df_sev.dropna(subset=["claim_amount"])

print("Baris sev dihapus karena claim_amount NaN:", rows_sev_before_claim - len(df_sev))


# =========================
# 6) Quick check jumlah baris akhir
# =========================
print("\nJumlah baris akhir freq:", len(df_freq_aug))
print("Jumlah baris akhir sev :", len(df_sev))

Baris freq dihapus karena exposure NaN: 415
Baris sev dihapus karena exposure NaN: 40
Baris sev dihapus karena claim_amount NaN: 34

Jumlah baris akhir freq: 94647
Jumlah baris akhir sev : 8198


In [ ]:
df_freq_aug.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94647 entries, 0 to 95061
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   policy_id        94424 non-null  object 
 1   equipment_id     94428 non-null  object 
 2   equipment_type   94408 non-null  object 
 3   equipment_age    47198 non-null  float64
 4   solar_system     94415 non-null  object 
 5   maintenance_int  94234 non-null  float64
 6   usage_int        94240 non-null  float64
 7   exposure         94647 non-null  float64
 8   claim_count      94647 non-null  int64  
dtypes: float64(4), int64(1), object(4)
memory usage: 7.2+ MB


In [ ]:
df_freq["solar_system"].value_counts(dropna=False)

,count
solar_system,
Epsilon,37926
Zeta,37807
Helionis Cluster,19095
NaN,234


In [ ]:
from sklearn.impute import SimpleImputer

# =========================
# Variabel kategorik yang akan diimpute
# =========================
cat_vars = ["solar_system", "equipment_type"]

# =========================
# Buat imputer mode
# =========================
mode_imputer = SimpleImputer(strategy="most_frequent")

# =========================
# Imputasi untuk sheet freq
# =========================
df_freq_aug[cat_vars] = mode_imputer.fit_transform(df_freq_aug[cat_vars])

print("Mode yang digunakan untuk freq:")
for i, col in enumerate(cat_vars):
    print(col, ":", mode_imputer.statistics_[i])

# =========================
# Imputasi untuk sheet sev
# =========================
mode_imputer_sev = SimpleImputer(strategy="most_frequent")

df_sev[cat_vars] = mode_imputer_sev.fit_transform(df_sev[cat_vars])

print("\nMode yang digunakan untuk sev:")
for i, col in enumerate(cat_vars):
    print(col, ":", mode_imputer_sev.statistics_[i])


# =========================
# Cek apakah masih ada missing
# =========================
print("\nMissing setelah imputasi (freq):")
print(df_freq_aug[cat_vars].isna().sum())

print("\nMissing setelah imputasi (sev):")
print(df_sev[cat_vars].isna().sum())

Mode yang digunakan untuk freq:
solar_system : Epsilon
equipment_type : ReglAggregators

Mode yang digunakan untuk sev:
solar_system : Epsilon
equipment_type : ReglAggregators

Missing setelah imputasi (freq):
solar_system      0
equipment_type    0
dtype: int64

Missing setelah imputasi (sev):
solar_system      0
equipment_type    0
dtype: int64


In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import OneHotEncoder

In [ ]:
numeric_vars = [
    "equipment_age",
    "maintenance_int",
    "usage_int",
    "exposure"
]

cat_vars = [
    "solar_system",
    "equipment_type"
]

In [ ]:
encoder = OneHotEncoder(drop="first", sparse_output=False)

# freq
freq_cat_encoded = encoder.fit_transform(df_freq_aug[cat_vars])
freq_cat_cols = encoder.get_feature_names_out(cat_vars)

df_freq_cat = pd.DataFrame(
    freq_cat_encoded,
    columns=freq_cat_cols,
    index=df_freq_aug.index
)

# sev
sev_cat_encoded = encoder.transform(df_sev[cat_vars])

df_sev_cat = pd.DataFrame(
    sev_cat_encoded,
    columns=freq_cat_cols,
    index=df_sev.index
)

In [ ]:
freq_impute_data = pd.concat(
    [df_freq_aug[numeric_vars], df_freq_cat],
    axis=1
)

sev_impute_data = pd.concat(
    [df_sev[numeric_vars], df_sev_cat],
    axis=1
)

In [ ]:
mice_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    random_state=42
)

In [ ]:
freq_imputed = mice_imputer.fit_transform(freq_impute_data)

freq_imputed_df = pd.DataFrame(
    freq_imputed,
    columns=freq_impute_data.columns,
    index=df_freq_aug.index
)

In [ ]:
sev_imputed = mice_imputer.fit_transform(sev_impute_data)

sev_imputed_df = pd.DataFrame(
    sev_imputed,
    columns=sev_impute_data.columns,
    index=df_sev.index
)

In [ ]:
for col in numeric_vars:
    df_freq_aug[col] = freq_imputed_df[col]
    df_sev[col] = sev_imputed_df[col]

In [ ]:
print("Missing numerik setelah MICE (freq)")
print(df_freq_aug[numeric_vars].isna().sum())

print("\nMissing numerik setelah MICE (sev)")
print(df_sev[numeric_vars].isna().sum())

Missing numerik setelah MICE (freq)
equipment_age      0
maintenance_int    0
usage_int          0
exposure           0
dtype: int64

Missing numerik setelah MICE (sev)
equipment_age      0
maintenance_int    0
usage_int          0
exposure           0
dtype: int64


In [ ]:
df_freq_aug.isna().sum()

,0
policy_id,223
equipment_id,219
equipment_type,0
equipment_age,0
solar_system,0
maintenance_int,0
usage_int,0
exposure,0
claim_count,0


In [ ]:
df_sev.isna().sum()

,0
claim_id,28
claim_seq,13
policy_id,22
equipment_id,15
equipment_type,0
equipment_age,0
solar_system,0
maintenance_int,0
usage_int,0
exposure,0


In [ ]:
# =========================
# EXPORT DATA FINAL KE EXCEL
# =========================

output_file = "equipment_failure_imputed_data.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    # sheet freq
    df_freq_aug.to_excel(
        writer,
        sheet_name="freq_final",
        index=False
    )

    # sheet sev
    df_sev.to_excel(
        writer,
        sheet_name="sev_final",
        index=False
    )

print("File berhasil disimpan:", output_file)

File berhasil disimpan: equipment_failure_imputed_data.xlsx


In [ ]:
from google.colab import files

files.download("equipment_failure_imputed_data.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>